# Análise dos Resultados

Análise exploratória com filtro **≥50% de zeros** no SVI:
remove tickers com 50% ou mais das semanas com SVI = 0.

Requer `previsoes_consolidadas.csv` gerado por `02_modelos.ipynb`.

In [10]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns

CAMINHO = "./data/"

NOMES = {
    
    "qlike_m0": "M0 (RW)",
    "qlike_m1": "M1 (GARCH)",
    "qlike_m2": "M2 (GARCH-X)",
}


In [11]:
# ── Carrega dados ────────────────────────────────────────────────
df_previsoes    = pd.read_csv(CAMINHO + "previsoes_consolidadas.csv", parse_dates=["semana"])
df_acoes        = pd.read_csv(CAMINHO + "acoes_elegiveis.csv")
df_final        = pd.read_csv(CAMINHO + "base_final_tcc.csv",        parse_dates=["semana"])
df_resumo_params = pd.read_csv(CAMINHO + "resumo_parametros.csv").set_index("ticker_b3")
df_trends       = pd.read_csv(CAMINHO + "google_trends_svi.csv",     parse_dates=["semana"])

In [ ]:
# ── Proporção de zeros no SVI ─────────────────────────────────────
zeros_pct = df_trends.groupby("ticker_b3")["svi"].apply(lambda x: (x == 0).mean())
df_resumo_params["svi_zeros_pct"] = zeros_pct.reindex(df_resumo_params.index)

df_resumo_params

In [ ]:
# ── Gráfico comparando as previsões ────────────────────────────────────────────────
TICKER = "PETR4"  # altere aqui para o ticker desejado

df_graf = (
    df_previsoes
    .query("ticker_b3 == @TICKER and split == 'out-of-sample'")
    .sort_values("semana")
)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df_graf["semana"], df_graf["rv"],         label="RV Realizada", color="black",   linewidth=2.0, zorder=5)
ax.plot(df_graf["semana"], df_graf["rv_pred_m0"], label="M0 (RW)",      color="#aec7e8", linewidth=1.2, linestyle="--")
ax.plot(df_graf["semana"], df_graf["rv_pred_m1"], label="M1 (GARCH)",   color="#ff7f0e", linewidth=1.2, linestyle="--")
ax.plot(df_graf["semana"], df_graf["rv_pred_m2"], label="M2 (GARCH-X)", color="#2ca02c", linewidth=1.2, linestyle="--")

ax.set_title(f"Previsões de Volatilidade — {TICKER} (OOS)", fontsize=13)
ax.set_xlabel("Semana")
ax.set_ylabel("Volatilidade Realizada")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Carrega dados ────────────────────────────────────────────────
df_previsoes    = pd.read_csv(CAMINHO + "previsoes_consolidadas.csv", parse_dates=["semana"])
df_acoes        = pd.read_csv(CAMINHO + "acoes_elegiveis.csv")
df_final        = pd.read_csv(CAMINHO + "base_final_tcc.csv",        parse_dates=["semana"])
df_resumo_params = pd.read_csv(CAMINHO + "resumo_parametros.csv").set_index("ticker_b3")
df_trends       = pd.read_csv(CAMINHO + "google_trends_svi.csv",     parse_dates=["semana"])

# ── Proporção de zeros no SVI ─────────────────────────────────────
zeros_pct = df_trends.groupby("ticker_b3")["svi"].apply(lambda x: (x == 0).mean())
df_resumo_params["svi_zeros_pct"] = zeros_pct

# ── Filtro ≥50%: remove tickers com ≥50% de zeros ────────────────
tickers_50 = zeros_pct[zeros_pct < 0.50].index
df_prev_50 = df_previsoes[df_previsoes["ticker_b3"].isin(tickers_50)]

# ── Remove erros de singular matrix ──────────────────────────────
tickers_erro = df_prev_50[df_prev_50["rv_pred_m2"].isna()]["ticker_b3"].unique()
df_prev_50   = df_prev_50[~df_prev_50["ticker_b3"].isin(tickers_erro)]

print(f"Tickers na análise: {df_prev_50['ticker_b3'].nunique()}")

# ── QLIKE ────────────────────────────────────────────────────────
def qlike(rv, pred):
    mask = (rv > 0) & (pred > 0)
    r    = np.where(mask, rv / pred, np.nan)
    return np.where(mask, r - np.log(r) - 1, np.nan)

df_oos = df_prev_50.query("split == 'out-of-sample'").copy()
df_oos["qlike_m0"] = qlike(df_oos["rv"], df_oos["rv_pred_m0"])
df_oos["qlike_m1"] = qlike(df_oos["rv"], df_oos["rv_pred_m1"])
df_oos["qlike_m2"] = qlike(df_oos["rv"], df_oos["rv_pred_m2"])

df_qlike = (
    df_oos
    .groupby("ticker_b3")[list(NOMES)]
    .mean()
    .rename(columns=NOMES)
)

oos_losses = (
    df_oos[list(NOMES)]
    .dropna()
    .rename(columns=NOMES)
    .reset_index(drop=True)
)

mcs = MCS(oos_losses, size=0.10, block_size=5, reps=1000, seed=42)
mcs.compute()
df_mcs = (
    mcs.pvalues
    .rename(columns={"Pvalue": "p_valor"})
    .assign(no_conjunto=lambda d: d["p_valor"] >= 0.10)
    .sort_values("p_valor", ascending=False)
)

df_qlike.to_csv(CAMINHO + "qlike_por_ticker.csv")
df_mcs.to_csv(CAMINHO + "mcs_resultado.csv")

print(f"\nQLIKE médio:")
print(oos_losses.mean().to_string())
print(f"\nMCS (α=10%):")
print(df_mcs.to_string())
print(f"\nConjunto de confiança: {list(mcs.included)}")
print(f"Excluídos           : {list(mcs.excluded)}")

# ── Resumo γ ──────────────────────────────────────────────────────
n_sig = (df_resumo_params["pct_signif_5pct"] >= 50).sum()
print(f"\nγ (ΔSVI) significativo em ≥50% das janelas:")
print(f"  {n_sig}/{len(df_resumo_params)} tickers")
print("\nTop 5 por % de janelas significativas:")
print(df_resumo_params["pct_signif_5pct"].sort_values(ascending=False).head().to_string())

In [ ]:
df_qlike["melhor"] = df_qlike[list(NOMES.values())].idxmin(axis=1)
print(df_qlike["melhor"].value_counts())

# Cruzar γ-significância com qual modelo vence por ticker
tickers_signif = df_resumo_params[df_resumo_params["pct_signif_5pct"] >= 50].index
# Filter tickers_signif to only include those present in df_qlike's index
tickers_signif = tickers_signif.intersection(df_qlike.index)
print(df_qlike.loc[tickers_signif, "melhor"].value_counts())

In [ ]:
# ── Grupos ─────────────────────────────────────────────────────
tickers_m2_vence = df_qlike[df_qlike["melhor"] == "M2 (GARCH-X)"].index
tickers_m2_perde = df_qlike[df_qlike["melhor"] != "M2 (GARCH-X)"].index
tickers_todos    = df_qlike.index

# ── Volume financeiro (R$) — fonte correta ─────────────────────
vol = (
    df_acoes
    .assign(ticker_b3=lambda d: d["ticker_yf"].str.replace(".SA", "", regex=False))
    .groupby("ticker_b3")["vol_fin_medio"]
    .first()
)

# ── Zeros no SVI ───────────────────────────────────────────────
zeros_svi = (
    df_final.groupby("ticker_b3")["svi"]
    .apply(lambda x: (x == 0).mean() * 100)
)

# ── Setor via yfinance ─────────────────────────────────────────
print("⏳ Buscando setor...")
setores = {}
for t in tickers_todos:
    try:
        setores[t] = yf.Ticker(t + ".SA").info.get("sector", "N/D")
    except:
        setores[t] = "N/D"

# ── DataFrame base ─────────────────────────────────────────────
df_analise = df_qlike[["melhor"]].copy()
df_analise["grupo"]        = np.where(df_analise.index.isin(tickers_m2_vence), "M2 vence", "M2 perde")
df_analise["zeros_pct"]    = zeros_svi
df_analise["vol_fin_medio"] = vol
df_analise["setor"]        = pd.Series(setores)

df_analise["zeros_bucket"] = pd.cut(
    df_analise["zeros_pct"],
    bins=range(0, 110, 10), right=False,
    labels=[f"{i}-{i+10}%" for i in range(0, 100, 10)]
)
df_analise["vol_bucket"] = pd.cut(
    df_analise["vol_fin_medio"],
    bins=[0, 1e6, 10e6, 100e6, 500e6, float("inf")],
    labels=["<1M", "1-10M", "10-100M", "100-500M", ">500M"]
)

# ── Função para montar tabela comparativa ──────────────────────
def tabela_comparativa(coluna, titulo):
    tab = (
        df_analise.groupby([coluna, "grupo"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["M2 perde", "M2 vence"], fill_value=0)
    )
    tab["Total"] = tab.sum(axis=1)
    tab.index.name = "Métrica"
    tab.columns.name = None
    print(f"\n{'═'*55}")
    print(f"  {titulo}")
    print(f"{'═'*55}")
    print(tab.to_string())

# ── Tabelas ────────────────────────────────────────────────────
tabela_comparativa("zeros_bucket", "1. Zeros no SVI")
tabela_comparativa("vol_bucket",   "2. Volume financeiro médio (R$)")
tabela_comparativa("setor",        "3. Setor")

# ── Mediana do volume por grupo ────────────────────────────────
print("\n  Mediana volume financeiro por grupo:")
print(
    df_analise.groupby("grupo")["vol_fin_medio"]
    .median()
    .apply(lambda x: f"R$ {x:,.0f}")
    .to_string()
)

In [ ]:
print(df_analise[df_analise["vol_bucket"] == ">500M"][["grupo", "vol_fin_medio"]].sort_values("vol_fin_medio", ascending=False))
print(df_analise.groupby("grupo")["zeros_pct"].median())

In [ ]:
# Verificar PDGR3 no df_acoes
print(df_acoes[df_acoes["ticker_yf"] == "PDGR3.SA"])

# E no df_final para ver o volume semanal
print(df_final[df_final["ticker_b3"] == "PDGR3"][["semana", "volume_total", "preco_fechamento"]].describe())

In [ ]:
import matplotlib.pyplot as plt

# % de tickers com svi=0 por semana, separado por grupo
df_final["grupo"] = np.where(
    df_final["ticker_b3"].isin(tickers_m2_vence), "M2 vence", "M2 perde"
)

zeros_semana = (
    df_final.groupby(["semana", "grupo"])
    .apply(lambda x: (x["svi"] == 0).mean() * 100)
    .reset_index(name="pct_zeros")
)

fig, ax = plt.subplots(figsize=(14, 5))

for grupo, cor in [("M2 vence", "#1f77b4"), ("M2 perde", "#d62728")]:
    d = zeros_semana[zeros_semana["grupo"] == grupo]
    ax.plot(d["semana"], d["pct_zeros"], label=grupo, color=cor, linewidth=1.5)

ax.set_title("% de tickers com SVI = 0 por semana", fontsize=13)
ax.set_xlabel("Semana")
ax.set_ylabel("% de tickers com SVI = 0")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()